# 3D Cleanroom CFD + Temperature Transport + Lagrangian Particles

This notebook extends the previous 2D cleanroom model into 3D.

It includes:

```text
3D incompressible airflow
3D temperature transport
3D inertial Lagrangian particles
Gauss-Seidel pressure solver
PyVista visualization
two separate animations
```

The two animations are:

```text
1. Particle movement + physical boundary setup only
2. Temperature field + physical boundary setup only
```

This is still an educational solver, not an industrial CFD solver.

## 1. Import libraries

This notebook uses:

```text
NumPy      = numerical arrays
PyVista    = 3D visualization
tqdm       = progress bars
Path       = save file paths
```

If PyVista is not installed, run:

```bash
pip install pyvista tqdm
```

If you are using Jupyter Notebook or JupyterLab, PyVista may also need:

```bash
pip install trame
```

In [51]:
import numpy as np
import pyvista as pv

from pathlib import Path
from tqdm.notebook import tqdm

# Try to make PyVista work inside Jupyter
try:
    pv.set_jupyter_backend("static")
except Exception:
    pass

## 2. 3D physical and numerical setup

The room is now a 3D box.

```text
x direction = room length
y direction = room width
z direction = room height/depth
```

The grid shape is:

```text
field[k, j, i]
```

meaning:

```text
i = x index
j = y index
k = z index
```

So the field arrays use:

```text
[z, y, x]
```

The incompressible flow condition is:

```text
div(u) = 0
```

or in 3D:

```text
du/dx + dv/dy + dw/dz = 0
```

The pressure equation is solved using a red-black Gauss-Seidel pressure solver.

In [52]:
# -----------------------------
# 3D geometry and numerical grid
# -----------------------------
Lx = 4.0
Ly = 4.0
Lz = 4.0

nx = 41
ny = 41
nz = 41

dx = Lx / (nx - 1)
dy = Ly / (ny - 1)
dz = Lz / (nz - 1)

x = np.linspace(0.0, Lx, nx)
y = np.linspace(0.0, Ly, ny)
z = np.linspace(0.0, Lz, nz)

# Field array shape:
# field[k, j, i] = field at z_k, y_j, x_i

# -----------------------------
# Fluid properties
# -----------------------------
rho = 1.0
nu = 0.03
U_in = 2.0

# -----------------------------
# Thermal / energy properties
# -----------------------------
rho_air = 1.2
cp_air = 1005.0
k_air = 0.0262

alpha = k_air / (rho_air * cp_air)

T_initial = 298.15
T_inlet = 308.15
T_aluminum = 298.15

T_body = 36.5 + 273.15

Q_T = 0.0

# -----------------------------
# CAD import settings
# -----------------------------
# Change this to your real CAD file.
# Supported examples: .stl, .obj, .ply, .vtk, .vtp
cad_file_path = Path(r"C:\Users\brian\OneDrive\桌面\ASE\cleanroom\simulation obstacles\tinker.obj")

# Scale your OBJ up so the longest side is about 4 m.
# For your tinker.obj, longest raw side is about 152 units.
cad_unit_scale = 4.0 / 152.0

# Pick a random 4 m × 4 m × 4 m region from the CAD model
cad_random_seed = 7

# "shell" is safer for most STL/OBJ files.
# "solid" only works well if CAD is a closed watertight solid.
cad_obstacle_mode = "shell"

# Thickness of CAD wall in grid mask when using shell mode
cad_surface_thickness = 0.75 * max(dx, dy, dz)

# -----------------------------
# Inlet geometry
# -----------------------------
inlet_diameter = 0.81
inlet_radius = inlet_diameter / 2.0
inlet_area = np.pi * inlet_radius**2

inlet_center_x = Lx / 2.0
inlet_center_y = Ly / 2.0

# -----------------------------
# Outlet strip geometry
# -----------------------------
outlet_floor_fraction = 0.17

outlet_strip_length_x = Lx
outlet_area = inlet_area * outlet_floor_fraction
outlet_strip_width_y = outlet_area / outlet_strip_length_x

outlet_x_min = 0.0
outlet_x_max = Lx

outlet_center_y = Ly / 2.0
outlet_y_min = outlet_center_y - outlet_strip_width_y / 2.0
outlet_y_max = outlet_center_y + outlet_strip_width_y / 2.0

outlet_flow_rate = 0.162
outlet_velocity = outlet_flow_rate / outlet_area

# -----------------------------
# Moving rectangular body
# -----------------------------
moving_body_length_x = 1.0
moving_body_width_y = 0.5
moving_body_height_z = 1.75

moving_body_speed = 1.5
moving_body_start_time = 1.0

moving_body_center_y = Ly / 2.0
moving_body_y_min = moving_body_center_y - moving_body_width_y / 2.0
moving_body_y_max = moving_body_center_y + moving_body_width_y / 2.0

moving_body_z_min = 0.0
moving_body_z_max = moving_body_height_z

moving_body_center_x_start = -moving_body_length_x / 2.0

moving_body_exit_time = (
    moving_body_start_time
    +
    (Lx + moving_body_length_x) / moving_body_speed
)

moving_body_release_margin = dx
particle_fall_release_vz = -0.25

# -----------------------------
# Lagrangian particle settings
# -----------------------------
max_particles = 50
release_every = 5
particles_per_release = 1
particle_substeps = 2

particle_release_z = Lz - 0.05

rng = np.random.default_rng(1)

particle_diameter_options = np.array([
    0.3e-6,
    0.5e-6,
    1.0e-6
])

particle_diameter_probabilities = np.array([
    0.70,
    0.24,
    0.06
])

particle_density = 1000.0
air_dynamic_viscosity = 1.8e-5

g_particle_x = 0.0
g_particle_y = 0.0
g_particle_z = -9.81

# -----------------------------
# Particle steady-state estimate settings
# -----------------------------
particle_speed_floor = 1.0e-4
particle_estimate_safety_factor = 1.25
particle_no_motion_required_checks = 3

# -----------------------------
# Optimized time-step setup
# -----------------------------
CFL_target = 0.20
diffusion_target = 0.25

U_ref_all = max(
    U_in,
    outlet_velocity,
    moving_body_speed
)

U_ref_x = U_ref_all
U_ref_y = U_ref_all
U_ref_z = U_ref_all

dt_cfl_x = CFL_target * dx / U_ref_x
dt_cfl_y = CFL_target * dy / U_ref_y
dt_cfl_z = CFL_target * dz / U_ref_z

inverse_grid_square_sum = (
    1.0 / dx**2
    +
    1.0 / dy**2
    +
    1.0 / dz**2
)

dt_velocity_diffusion_safe = diffusion_target / (
    nu * inverse_grid_square_sum
)

dt_thermal_diffusion_safe = diffusion_target / (
    alpha * inverse_grid_square_sum
)

dt = min(
    dt_cfl_x,
    dt_cfl_y,
    dt_cfl_z,
    dt_velocity_diffusion_safe,
    dt_thermal_diffusion_safe
)+0.02

nt = 1000
pressure_iterations = 40
frame_interval = 15

# -----------------------------
# Steady-state detection settings
# -----------------------------
steady_check_interval = 20

temperature_steady_tolerance = 1.0e-3
temperature_steady_required_checks = 5

particle_steady_required_checks = particle_no_motion_required_checks

stop_when_both_steady = True

# -----------------------------
# Save folder
# -----------------------------
save_folder = Path(r"C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation")
save_folder.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Final stability quantities
# -----------------------------
CFL_x = U_ref_x * dt / dx
CFL_y = U_ref_y * dt / dy
CFL_z = U_ref_z * dt / dz

velocity_diffusion_number = nu * dt * inverse_grid_square_sum
thermal_diffusion_number = alpha * dt * inverse_grid_square_sum

physical_simulation_time = nt * dt

particle_tau_options = (
    particle_density * particle_diameter_options**2
    /
    (18.0 * air_dynamic_viscosity)
)

inlet_flow_rate = U_in * inlet_area

# -----------------------------
# Print setup summary
# -----------------------------
print("3D grid setup")
print(f"nx = {nx}")
print(f"ny = {ny}")
print(f"nz = {nz}")
print(f"dx = {dx:.6f} m")
print(f"dy = {dy:.6f} m")
print(f"dz = {dz:.6f} m")

print()
print("Flow direction")
print("Inlet face: upper z face, z = Lz")
print("Outlet face: lower z face, z = 0")
print("Main inlet velocity direction: -z")

print()
print("CAD boundary setup")
print(f"CAD file path = {cad_file_path}")
print(f"CAD unit scale = {cad_unit_scale}")
print(f"CAD obstacle mode = {cad_obstacle_mode}")
print(f"CAD random seed = {cad_random_seed}")
print(f"CAD surface thickness = {cad_surface_thickness:.4f} m")

print()
print("Outlet strip")
print(f"Inlet area = {inlet_area:.6f} m²")
print(f"Outlet area = inlet area * 0.17 = {outlet_area:.6f} m²")
print(f"Outlet strip length x = {outlet_strip_length_x:.3f} m")
print(f"Outlet strip width y = {outlet_strip_width_y:.6f} m")
print(f"Outlet pumping flow rate = {outlet_flow_rate:.3f} m³/s")
print(f"Outlet velocity magnitude = {outlet_velocity:.3f} m/s")
print(f"Inlet flow rate from U_in = {inlet_flow_rate:.3f} m³/s")

print()
print("Moving body")
print(f"Body size = {moving_body_length_x:.2f} m × {moving_body_width_y:.2f} m × {moving_body_height_z:.2f} m")
print(f"Body speed = {moving_body_speed:.2f} m/s")
print(f"Body temperature = {T_body - 273.15:.1f} °C")
print(f"Body start time = {moving_body_start_time:.3f} s")
print(f"Body exit time = {moving_body_exit_time:.3f} s")

print()
print("Simulation setup")
print(f"dt = {dt:.6f} s")
print(f"nt = {nt}")
print(f"Physical simulation time = {physical_simulation_time:.3f} s")
print(f"pressure_iterations = {pressure_iterations}")
print(f"Maximum particles = {max_particles}")

3D grid setup
nx = 41
ny = 41
nz = 41
dx = 0.100000 m
dy = 0.100000 m
dz = 0.100000 m

Flow direction
Inlet face: upper z face, z = Lz
Outlet face: lower z face, z = 0
Main inlet velocity direction: -z

CAD boundary setup
CAD file path = C:\Users\brian\OneDrive\桌面\ASE\cleanroom\simulation obstacles\tinker.obj
CAD unit scale = 0.02631578947368421
CAD obstacle mode = shell
CAD random seed = 7
CAD surface thickness = 0.0750 m

Outlet strip
Inlet area = 0.515300 m²
Outlet area = inlet area * 0.17 = 0.087601 m²
Outlet strip length x = 4.000 m
Outlet strip width y = 0.021900 m
Outlet pumping flow rate = 0.162 m³/s
Outlet velocity magnitude = 1.849 m/s
Inlet flow rate from U_in = 1.031 m³/s

Moving body
Body size = 1.00 m × 0.50 m × 1.75 m
Body speed = 1.50 m/s
Body temperature = 36.5 °C
Body start time = 1.000 s
Body exit time = 4.333 s

Simulation setup
dt = 0.030000 s
nt = 1000
Physical simulation time = 30.000 s
pressure_iterations = 40
Maximum particles = 50


## 3. 3D fields, inlet/outlet, and rectangular obstacles

The unknown fields are:

```text
u = x-direction velocity
v = y-direction velocity
w = z-direction velocity
p = pressure
T = temperature
```

The inlet is a circular disk on the left wall:

```text
x = 0
diameter = 0.81 m
center = (y = 2.0 m, z = 2.0 m)
```

The outlet is a circular disk on the right wall:

```text
x = 4.0 m
diameter = 0.34 m
center = (y = 2.0 m, z = 2.0 m)
```

The spherical/circular obstacle is removed.

Instead, there are two rectangular aluminum obstacles attached to the right wall.

Each obstacle has:

```text
x length = 2.0 m
y thickness = 1.0 m
z depth = full room depth
```

So each obstacle is a rectangular prism.

Their y-centers are:

```text
top obstacle center y = 3.09 m
bottom obstacle center y = 0.91 m
```

In [53]:
# -----------------------------
# 3D flow fields
# Shape: [z, y, x]
# -----------------------------
u_field = np.zeros((nz, ny, nx))
v_field = np.zeros((nz, ny, nx))
w_field = np.zeros((nz, ny, nx))
p = np.zeros((nz, ny, nx))

T_field = T_initial * np.ones((nz, ny, nx))

# -----------------------------
# Inlet / outlet masks on z-faces
# Face shape: [y, x]
# -----------------------------
X_face, Y_face = np.meshgrid(x, y)

inlet = (
    (X_face - inlet_center_x)**2
    +
    (Y_face - inlet_center_y)**2
) <= inlet_radius**2

outlet = (
    (X_face >= outlet_x_min)
    &
    (X_face <= outlet_x_max)
    &
    (Y_face >= outlet_y_min)
    &
    (Y_face <= outlet_y_max)
)

# -----------------------------
# CAD import helper functions
# -----------------------------
def load_and_prepare_cad_mesh(cad_path, unit_scale):
    """
    Load CAD mesh, scale it, extract surface, triangulate, and clean.
    """

    if not cad_path.exists():
        raise FileNotFoundError(
            f"CAD file not found:\n{cad_path}\n\n"
            "Please change cad_file_path in Cell 5."
        )

    mesh = pv.read(str(cad_path))

    if isinstance(mesh, pv.MultiBlock):
        mesh = mesh.combine()

    mesh = mesh.extract_surface().triangulate().clean()

    if unit_scale != 1.0:
        mesh.points *= unit_scale

    return mesh


def choose_random_4m_cube_from_cad(mesh, seed):
    """
    Pick a 4 m × 4 m × 4 m region from the CAD.

    This safer version first chooses an actual CAD point,
    then builds the 4 m box around that point.
    """

    rng_local = np.random.default_rng(seed)

    pts = mesh.points

    if pts.shape[0] == 0:
        raise RuntimeError("CAD mesh has no points.")

    # Randomly pick one real CAD point
    random_point = pts[rng_local.integers(0, pts.shape[0])]

    target_length = np.array([Lx, Ly, Lz])

    cube_min = random_point - target_length / 2.0
    cube_max = cube_min + target_length

    xmin, xmax, ymin, ymax, zmin, zmax = mesh.bounds

    cad_min = np.array([xmin, ymin, zmin])
    cad_max = np.array([xmax, ymax, zmax])
    cad_length = cad_max - cad_min

    # Clamp box if CAD is larger than 4 m in that direction
    for d in range(3):
        if cad_length[d] >= target_length[d]:

            if cube_min[d] < cad_min[d]:
                cube_min[d] = cad_min[d]
                cube_max[d] = cube_min[d] + target_length[d]

            if cube_max[d] > cad_max[d]:
                cube_max[d] = cad_max[d]
                cube_min[d] = cube_max[d] - target_length[d]

    selected_bounds = (
        cube_min[0], cube_max[0],
        cube_min[1], cube_max[1],
        cube_min[2], cube_max[2]
    )

    return selected_bounds


def crop_cad_to_selected_bounds(mesh, selected_bounds):
    """
    Crop CAD to the selected 4 m box.

    This version uses point extraction directly.
    Since choose_random_4m_cube_from_cad picked a real CAD point,
    this should not be empty.
    """

    xmin, xmax, ymin, ymax, zmin, zmax = selected_bounds

    pts = mesh.points

    point_inside = (
        (pts[:, 0] >= xmin)
        &
        (pts[:, 0] <= xmax)
        &
        (pts[:, 1] >= ymin)
        &
        (pts[:, 1] <= ymax)
        &
        (pts[:, 2] >= zmin)
        &
        (pts[:, 2] <= zmax)
    )

    if not np.any(point_inside):
        raise RuntimeError(
            "The selected CAD region still contains no CAD points. "
            "Try changing cad_random_seed, or check cad_unit_scale."
        )

    cropped = mesh.extract_points(
        point_inside,
        adjacent_cells=True,
        include_cells=True
    )

    cropped = cropped.extract_surface().triangulate().clean()

    return cropped


def map_cad_crop_to_simulation_domain(cropped_mesh, selected_bounds):
    """
    Move selected CAD cube into simulation coordinates:
    selected CAD cube -> [0,4] × [0,4] × [0,4]
    """

    xmin, xmax, ymin, ymax, zmin, zmax = selected_bounds

    sim_mesh = cropped_mesh.copy()

    sim_mesh.points[:, 0] -= xmin
    sim_mesh.points[:, 1] -= ymin
    sim_mesh.points[:, 2] -= zmin

    return sim_mesh.extract_surface().triangulate().clean()


def clear_inlet_outlet_from_obstacle_mask(mask):
    """
    Do not allow imported CAD obstacle to block inlet and outlet openings.
    """

    top_face = mask[-1, :, :].copy()
    top_face[inlet] = False
    mask[-1, :, :] = top_face

    bottom_face = mask[0, :, :].copy()
    bottom_face[outlet] = False
    mask[0, :, :] = bottom_face

    return mask


def make_cad_obstacle_mask(sim_mesh):
    """
    Convert CAD mesh into fixed_obstacle mask with shape [z, y, x].
    """

    Xg, Yg, Zg = np.meshgrid(
        x,
        y,
        z,
        indexing="ij"
    )

    points = np.column_stack(
        (
            Xg.ravel(order="C"),
            Yg.ravel(order="C"),
            Zg.ravel(order="C")
        )
    )

    point_cloud = pv.PolyData(points)
    surface = sim_mesh.extract_surface().triangulate().clean()

    if cad_obstacle_mode.lower() == "solid":
        selected = point_cloud.select_enclosed_points(
            surface,
            tolerance=0.0,
            check_surface=False
        )

        solid_flat = selected.point_data["SelectedPoints"].astype(bool)

    else:
        distance_data = point_cloud.compute_implicit_distance(surface)
        distance = distance_data.point_data["implicit_distance"]

        solid_flat = np.abs(distance) <= cad_surface_thickness

    solid_xyz = solid_flat.reshape((nx, ny, nz), order="C")

    # Convert [x, y, z] to solver shape [z, y, x]
    solid_zyx = np.transpose(solid_xyz, (2, 1, 0))

    solid_zyx = clear_inlet_outlet_from_obstacle_mask(solid_zyx)

    return solid_zyx


# -----------------------------
# Load CAD and create fixed_obstacle
# -----------------------------
cad_full_mesh = load_and_prepare_cad_mesh(
    cad_file_path,
    cad_unit_scale
)

cad_selected_bounds = choose_random_4m_cube_from_cad(
    cad_full_mesh,
    cad_random_seed
)

cad_cropped_mesh = crop_cad_to_selected_bounds(
    cad_full_mesh,
    cad_selected_bounds
)

cad_obstacle_visual_mesh = map_cad_crop_to_simulation_domain(
    cad_cropped_mesh,
    cad_selected_bounds
)

fixed_obstacle = make_cad_obstacle_mask(
    cad_obstacle_visual_mesh
)

# -----------------------------
# Moving body functions
# -----------------------------
X3 = x[None, None, :]
Y3 = y[None, :, None]
Z3 = z[:, None, None]


def get_moving_body_state(time_value):
    """
    Moving body is inactive before 1 second,
    moves once from left to right,
    then becomes inactive after leaving.
    """

    if time_value < moving_body_start_time:
        return None, 0.0, False

    elapsed_time = time_value - moving_body_start_time

    center_x = (
        moving_body_center_x_start
        +
        moving_body_speed * elapsed_time
    )

    x_min_body = center_x - moving_body_length_x / 2.0
    x_max_body = center_x + moving_body_length_x / 2.0

    body_active = (
        x_max_body >= 0.0
        and
        x_min_body <= Lx
    )

    if not body_active:
        return None, 0.0, False

    bounds = (
        x_min_body,
        x_max_body,
        moving_body_y_min,
        moving_body_y_max,
        moving_body_z_min,
        moving_body_z_max
    )

    return bounds, moving_body_speed, True


def get_moving_body_mask(time_value):
    """
    Return moving body mask.
    """

    body_bounds, body_velocity_x, body_active = get_moving_body_state(time_value)

    if not body_active:
        body_mask = np.zeros((nz, ny, nx), dtype=bool)
        return body_mask, None, 0.0, False

    x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

    body_mask = (
        (X3 >= x_min_body)
        &
        (X3 <= x_max_body)
        &
        (Y3 >= y_min_body)
        &
        (Y3 <= y_max_body)
        &
        (Z3 >= z_min_body)
        &
        (Z3 <= z_max_body)
    )

    return body_mask, body_bounds, body_velocity_x, body_active


initial_body_mask, initial_body_bounds, initial_body_velocity_x, initial_body_active = get_moving_body_mask(0.0)

solid_mask = fixed_obstacle | initial_body_mask
fluid = ~solid_mask

T_field[fixed_obstacle] = T_aluminum

if initial_body_active:
    T_field[initial_body_mask] = T_body

# -----------------------------
# Red-black patterns for pressure solver
# -----------------------------
K_i, J_i, I_i = np.indices((nz - 2, ny - 2, nx - 2))

red_pattern = ((I_i + J_i + K_i) % 2 == 0)
black_pattern = ((I_i + J_i + K_i) % 2 == 1)

print("CAD imported successfully.")
print()
print("Original CAD bounds after scaling:")
print(cad_full_mesh.bounds)

print()
print("Random selected CAD bounds:")
print(cad_selected_bounds)

print()
print("Selected CAD region mapped into 4m × 4m × 4m domain:")
print(cad_obstacle_visual_mesh.bounds)

print()
print("Inlet face grid points:", np.sum(inlet))
print("Outlet face grid points:", np.sum(outlet))
print("CAD fixed obstacle grid points:", np.sum(fixed_obstacle))
print("Fluid grid points:", np.sum(fluid))
print("Initial moving body active:", initial_body_active)

C:\Users\brian\AppData\Local\Temp\ipykernel_19008\769973827.py:53: PyVistaFutureWarning: The default value of `algorithm` for the filter
`PolyData.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  mesh = mesh.extract_surface().triangulate().clean()
C:\Users\brian\AppData\Local\Temp\ipykernel_19008\769973827.py:150: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  cropped = cropped.extract_surface().triangulate().clean()
C:\Users\brian\AppData\Local\Temp\ipykernel_19008\769973827.py:169: PyVistaFutureWarning: The default value of `algorithm` for the filter
`PolyData.extract_surface` will change in the future. It currently defaults to
`'da

CAD imported successfully.

Original CAD bounds after scaling:
BoundsTuple(x_min = -1.0,
            x_max =  0.7894736842105263,
            y_min = -3.0,
            y_max =  3.4473684210526314,
            z_min = -3.9473684210526314,
            z_max =  0.0)

Random selected CAD bounds:
(np.float64(-2.9210526315789473), np.float64(1.0789473684210527), np.float64(-2.3684210526315788), np.float64(1.6315789473684212), np.float64(-5.947368421052632), np.float64(-1.9473684210526319))

Selected CAD region mapped into 4m × 4m × 4m domain:
BoundsTuple(x_min =  1.9210526315789473,
            x_max =  3.7105263157894735,
            y_min = -0.6315789473684212,
            y_max =  5.815789473684211,
            z_min =  2.0000000000000004,
            z_max =  5.947368421052632)

Inlet face grid points: 49
Outlet face grid points: 41
CAD fixed obstacle grid points: 2721
Fluid grid points: 66200
Initial moving body active: False


## 4. Boundary conditions

Velocity boundary conditions:

```text
Room walls: no-slip
Inlet: fixed x-velocity
Outlet: zero-gradient velocity
Obstacles: no-slip solid
```

Pressure boundary conditions:

```text
Room walls: zero normal pressure gradient
Outlet: reference pressure p = 0
Obstacles: approximate zero normal pressure gradient
```

Temperature boundary conditions:

```text
Room walls: zero-flux temperature
Inlet: fixed temperature
Outlet: zero-gradient temperature
Obstacles: constant aluminum temperature
```

In [54]:
def average_inside_solid_from_neighbors(field, mask):
    """
    Fill solid values using neighbor averages.
    This is only a simple educational approximation.
    """

    avg = np.zeros_like(field)
    count = np.zeros_like(field)

    avg[1:, :, :] += field[:-1, :, :]
    count[1:, :, :] += 1.0

    avg[:-1, :, :] += field[1:, :, :]
    count[:-1, :, :] += 1.0

    avg[:, 1:, :] += field[:, :-1, :]
    count[:, 1:, :] += 1.0

    avg[:, :-1, :] += field[:, 1:, :]
    count[:, :-1, :] += 1.0

    avg[:, :, 1:] += field[:, :, :-1]
    count[:, :, 1:] += 1.0

    avg[:, :, :-1] += field[:, :, 1:]
    count[:, :, :-1] += 1.0

    neighbor_average = avg / np.maximum(count, 1.0)
    field[mask] = neighbor_average[mask]

    return field


def apply_velocity_bcs(u, v, w, solid_mask, body_mask, body_velocity_x):
    """
    3D velocity boundary conditions.

    Inlet:
        upper z face, z = Lz
        velocity points downward, w = -U_in

    Outlet:
        lower z face, z = 0
        prescribed pumping velocity from outlet flow rate

    Moving body:
        u = body_velocity_x, v = 0, w = 0
    """

    # x walls
    u[:, :, 0] = 0.0
    v[:, :, 0] = 0.0
    w[:, :, 0] = 0.0

    u[:, :, -1] = 0.0
    v[:, :, -1] = 0.0
    w[:, :, -1] = 0.0

    # y walls
    u[:, 0, :] = 0.0
    v[:, 0, :] = 0.0
    w[:, 0, :] = 0.0

    u[:, -1, :] = 0.0
    v[:, -1, :] = 0.0
    w[:, -1, :] = 0.0

    # lower z wall
    u[0, :, :] = 0.0
    v[0, :, :] = 0.0
    w[0, :, :] = 0.0

    # upper z wall
    u[-1, :, :] = 0.0
    v[-1, :, :] = 0.0
    w[-1, :, :] = 0.0

    # Inlet on upper z face
    upper_u = u[-1, :, :].copy()
    upper_v = v[-1, :, :].copy()
    upper_w = w[-1, :, :].copy()

    upper_u[inlet] = 0.0
    upper_v[inlet] = 0.0
    upper_w[inlet] = -U_in

    u[-1, :, :] = upper_u
    v[-1, :, :] = upper_v
    w[-1, :, :] = upper_w

    # Outlet pumping on lower z face
    lower_u = u[0, :, :].copy()
    lower_v = v[0, :, :].copy()
    lower_w = w[0, :, :].copy()

    lower_u[outlet] = 0.0
    lower_v[outlet] = 0.0
    lower_w[outlet] = -outlet_velocity

    u[0, :, :] = lower_u
    v[0, :, :] = lower_v
    w[0, :, :] = lower_w

    # Fixed obstacles
    u[fixed_obstacle] = 0.0
    v[fixed_obstacle] = 0.0
    w[fixed_obstacle] = 0.0

    # Moving body
    u[body_mask] = body_velocity_x
    v[body_mask] = 0.0
    w[body_mask] = 0.0

    return u, v, w


def apply_pressure_bcs(p_field, solid_mask):
    """
    3D pressure boundary conditions.
    """

    # x walls
    p_field[:, :, 0] = p_field[:, :, 1]
    p_field[:, :, -1] = p_field[:, :, -2]

    # y walls
    p_field[:, 0, :] = p_field[:, 1, :]
    p_field[:, -1, :] = p_field[:, -2, :]

    # z walls
    p_field[0, :, :] = p_field[1, :, :]
    p_field[-1, :, :] = p_field[-2, :, :]

    # Outlet reference pressure on lower z face
    lower_p = p_field[0, :, :].copy()
    lower_p[outlet] = 0.0
    p_field[0, :, :] = lower_p

    p_field = average_inside_solid_from_neighbors(p_field, solid_mask)

    return p_field


def apply_temperature_bcs(T, solid_mask, body_mask):
    """
    3D temperature boundary conditions.

    Inlet:
        upper z face, fixed T = T_inlet

    Outlet:
        lower z face outlet strip, zero-gradient temperature

    Fixed obstacles:
        T = T_aluminum

    Moving body:
        T = T_body
    """

    # x wall zero-flux
    T[:, :, 0] = T[:, :, 1]
    T[:, :, -1] = T[:, :, -2]

    # y wall zero-flux
    T[:, 0, :] = T[:, 1, :]
    T[:, -1, :] = T[:, -2, :]

    # z wall zero-flux
    T[0, :, :] = T[1, :, :]
    T[-1, :, :] = T[-2, :, :]

    # Fixed inlet temperature on upper z face
    upper_T = T[-1, :, :].copy()
    upper_T[inlet] = T_inlet
    T[-1, :, :] = upper_T

    # Outlet zero-gradient on lower z face
    lower_T = T[0, :, :].copy()
    lower_T[outlet] = T[1, :, :][outlet]
    T[0, :, :] = lower_T

    # Fixed obstacles
    T[fixed_obstacle] = T_aluminum

    # Moving body
    T[body_mask] = T_body

    return T

## 5. Pressure Poisson equation with Gauss-Seidel solver

The incompressibility condition is:

```text
div(u) = 0
```

In the projection method, the solver first predicts a temporary velocity field:

```text
u*
v*
w*
```

Then pressure is solved from:

```text
∇²p = rho / dt · div(u*)
```

In 3D:

```text
∇²p = d²p/dx² + d²p/dy² + d²p/dz²
```

This notebook uses a red-black Gauss-Seidel method instead of the older Jacobi-style pressure solver.

Gauss-Seidel usually converges faster because it immediately uses updated pressure values.

In [55]:
def build_pressure_rhs(u_star, v_star, w_star, solid_mask):
    """
    Build RHS for 3D pressure Poisson equation.

    b = rho/dt * div(u*)
    """

    b = np.zeros_like(p)

    b[1:-1, 1:-1, 1:-1] = (rho / dt) * (
        (u_star[1:-1, 1:-1, 2:] - u_star[1:-1, 1:-1, :-2]) / (2.0 * dx)
        +
        (v_star[1:-1, 2:, 1:-1] - v_star[1:-1, :-2, 1:-1]) / (2.0 * dy)
        +
        (w_star[2:, 1:-1, 1:-1] - w_star[:-2, 1:-1, 1:-1]) / (2.0 * dz)
    )

    b[solid_mask] = 0.0

    return b


def pressure_candidate_3d(p_field, b):
    """
    Candidate pressure update for 3D Poisson equation.
    """

    dx2 = dx**2
    dy2 = dy**2
    dz2 = dz**2

    denominator = 2.0 * (
        dy2 * dz2
        +
        dx2 * dz2
        +
        dx2 * dy2
    )

    candidate = (
        (p_field[1:-1, 1:-1, 2:] + p_field[1:-1, 1:-1, :-2]) * dy2 * dz2
        +
        (p_field[1:-1, 2:, 1:-1] + p_field[1:-1, :-2, 1:-1]) * dx2 * dz2
        +
        (p_field[2:, 1:-1, 1:-1] + p_field[:-2, 1:-1, 1:-1]) * dx2 * dy2
        -
        b[1:-1, 1:-1, 1:-1] * dx2 * dy2 * dz2
    ) / denominator

    return candidate


def solve_pressure_gauss_seidel(p_field, b, solid_mask):
    """
    Red-black Gauss-Seidel pressure solver.
    The moving body changes the fluid mask every time step.
    """

    fluid_interior_mask = (~solid_mask)[1:-1, 1:-1, 1:-1]

    red_mask = red_pattern & fluid_interior_mask
    black_mask = black_pattern & fluid_interior_mask

    for _ in range(pressure_iterations):

        # Red update
        candidate = pressure_candidate_3d(p_field, b)
        center = p_field[1:-1, 1:-1, 1:-1]
        center[red_mask] = candidate[red_mask]
        p_field[1:-1, 1:-1, 1:-1] = center

        p_field = apply_pressure_bcs(p_field, solid_mask)

        # Black update
        candidate = pressure_candidate_3d(p_field, b)
        center = p_field[1:-1, 1:-1, 1:-1]
        center[black_mask] = candidate[black_mask]
        p_field[1:-1, 1:-1, 1:-1] = center

        p_field = apply_pressure_bcs(p_field, solid_mask)

    return p_field

## 6. 3D incompressible velocity-pressure step

Each time step does:

```text
1. Predict velocity without pressure
2. Solve pressure using Gauss-Seidel
3. Correct velocity using pressure gradient
4. Re-apply boundary conditions
```

The momentum equation is solved in simplified form:

```text
du/dt + u du/dx + v du/dy + w du/dz = -1/rho dp/dx + nu ∇²u
```

and similarly for:

```text
v velocity
w velocity
```

In [56]:
def step_velocity_pressure_3d(
    u,
    v,
    w,
    p_field,
    solid_mask,
    body_mask,
    body_velocity_x
):
    """
    Advance 3D incompressible velocity-pressure field by one time step.
    """

    u = u.copy()
    v = v.copy()
    w = w.copy()

    un = u.copy()
    vn = v.copy()
    wn = w.copy()

    u_star = un.copy()
    v_star = vn.copy()
    w_star = wn.copy()

    fluid_interior = (~solid_mask)[1:-1, 1:-1, 1:-1]

    # -----------------------------
    # Derivatives
    # -----------------------------
    du_dx = (un[1:-1, 1:-1, 2:] - un[1:-1, 1:-1, :-2]) / (2.0 * dx)
    du_dy = (un[1:-1, 2:, 1:-1] - un[1:-1, :-2, 1:-1]) / (2.0 * dy)
    du_dz = (un[2:, 1:-1, 1:-1] - un[:-2, 1:-1, 1:-1]) / (2.0 * dz)

    dv_dx = (vn[1:-1, 1:-1, 2:] - vn[1:-1, 1:-1, :-2]) / (2.0 * dx)
    dv_dy = (vn[1:-1, 2:, 1:-1] - vn[1:-1, :-2, 1:-1]) / (2.0 * dy)
    dv_dz = (vn[2:, 1:-1, 1:-1] - vn[:-2, 1:-1, 1:-1]) / (2.0 * dz)

    dw_dx = (wn[1:-1, 1:-1, 2:] - wn[1:-1, 1:-1, :-2]) / (2.0 * dx)
    dw_dy = (wn[1:-1, 2:, 1:-1] - wn[1:-1, :-2, 1:-1]) / (2.0 * dy)
    dw_dz = (wn[2:, 1:-1, 1:-1] - wn[:-2, 1:-1, 1:-1]) / (2.0 * dz)

    # -----------------------------
    # Laplacians
    # -----------------------------
    lap_u = (
        (un[1:-1, 1:-1, 2:] - 2.0 * un[1:-1, 1:-1, 1:-1] + un[1:-1, 1:-1, :-2]) / dx**2
        +
        (un[1:-1, 2:, 1:-1] - 2.0 * un[1:-1, 1:-1, 1:-1] + un[1:-1, :-2, 1:-1]) / dy**2
        +
        (un[2:, 1:-1, 1:-1] - 2.0 * un[1:-1, 1:-1, 1:-1] + un[:-2, 1:-1, 1:-1]) / dz**2
    )

    lap_v = (
        (vn[1:-1, 1:-1, 2:] - 2.0 * vn[1:-1, 1:-1, 1:-1] + vn[1:-1, 1:-1, :-2]) / dx**2
        +
        (vn[1:-1, 2:, 1:-1] - 2.0 * vn[1:-1, 1:-1, 1:-1] + vn[1:-1, :-2, 1:-1]) / dy**2
        +
        (vn[2:, 1:-1, 1:-1] - 2.0 * vn[1:-1, 1:-1, 1:-1] + vn[:-2, 1:-1, 1:-1]) / dz**2
    )

    lap_w = (
        (wn[1:-1, 1:-1, 2:] - 2.0 * wn[1:-1, 1:-1, 1:-1] + wn[1:-1, 1:-1, :-2]) / dx**2
        +
        (wn[1:-1, 2:, 1:-1] - 2.0 * wn[1:-1, 1:-1, 1:-1] + wn[1:-1, :-2, 1:-1]) / dy**2
        +
        (wn[2:, 1:-1, 1:-1] - 2.0 * wn[1:-1, 1:-1, 1:-1] + wn[:-2, 1:-1, 1:-1]) / dz**2
    )

    # -----------------------------
    # Predict velocity
    # -----------------------------
    predicted_u = un[1:-1, 1:-1, 1:-1] + dt * (
        -(un[1:-1, 1:-1, 1:-1] * du_dx
          + vn[1:-1, 1:-1, 1:-1] * du_dy
          + wn[1:-1, 1:-1, 1:-1] * du_dz)
        +
        nu * lap_u
    )

    predicted_v = vn[1:-1, 1:-1, 1:-1] + dt * (
        -(un[1:-1, 1:-1, 1:-1] * dv_dx
          + vn[1:-1, 1:-1, 1:-1] * dv_dy
          + wn[1:-1, 1:-1, 1:-1] * dv_dz)
        +
        nu * lap_v
    )

    predicted_w = wn[1:-1, 1:-1, 1:-1] + dt * (
        -(un[1:-1, 1:-1, 1:-1] * dw_dx
          + vn[1:-1, 1:-1, 1:-1] * dw_dy
          + wn[1:-1, 1:-1, 1:-1] * dw_dz)
        +
        nu * lap_w
    )

    u_star[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, predicted_u, 0.0)
    v_star[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, predicted_v, 0.0)
    w_star[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, predicted_w, 0.0)

    u_star, v_star, w_star = apply_velocity_bcs(
        u_star,
        v_star,
        w_star,
        solid_mask,
        body_mask,
        body_velocity_x
    )

    # -----------------------------
    # Pressure solve
    # -----------------------------
    b = build_pressure_rhs(u_star, v_star, w_star, solid_mask)
    p_field = solve_pressure_gauss_seidel(p_field, b, solid_mask)

    # -----------------------------
    # Correct velocity
    # -----------------------------
    u_new = u_star.copy()
    v_new = v_star.copy()
    w_new = w_star.copy()

    corrected_u = u_star[1:-1, 1:-1, 1:-1] - (dt / rho) * (
        (p_field[1:-1, 1:-1, 2:] - p_field[1:-1, 1:-1, :-2]) / (2.0 * dx)
    )

    corrected_v = v_star[1:-1, 1:-1, 1:-1] - (dt / rho) * (
        (p_field[1:-1, 2:, 1:-1] - p_field[1:-1, :-2, 1:-1]) / (2.0 * dy)
    )

    corrected_w = w_star[1:-1, 1:-1, 1:-1] - (dt / rho) * (
        (p_field[2:, 1:-1, 1:-1] - p_field[:-2, 1:-1, 1:-1]) / (2.0 * dz)
    )

    u_new[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, corrected_u, 0.0)
    v_new[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, corrected_v, 0.0)
    w_new[1:-1, 1:-1, 1:-1] = np.where(fluid_interior, corrected_w, 0.0)

    u_new, v_new, w_new = apply_velocity_bcs(
        u_new,
        v_new,
        w_new,
        solid_mask,
        body_mask,
        body_velocity_x
    )

    return u_new, v_new, w_new, p_field

## 7. 3D temperature transport equation

The temperature-based energy equation is:

```text
rho cp (dT/dt + u dT/dx + v dT/dy + w dT/dz) = div(k grad(T)) + Q
```

Assuming constant thermal conductivity:

```text
grad(k) = 0
```

so:

```text
div(k grad(T)) = k ∇²T
```

After dividing by `rho cp`:

```text
dT/dt + u dT/dx + v dT/dy + w dT/dz = alpha ∇²T + Q/(rho cp)
```

where:

```text
alpha = k / (rho cp)
```

The flow is incompressible, so the conservative and advective forms are equivalent.

In [57]:
def step_temperature_3d(T, u, v, w, solid_mask, body_mask):
    """
    Advance 3D temperature field by one step.

    dT/dt + u dT/dx + v dT/dy + w dT/dz
    =
    alpha ∇²T + Q/(rho cp)
    """

    T = T.copy()
    Tn = T.copy()

    fluid_interior = (~solid_mask)[1:-1, 1:-1, 1:-1]

    # Upwind derivatives
    dTdx_backward = (Tn[1:-1, 1:-1, 1:-1] - Tn[1:-1, 1:-1, :-2]) / dx
    dTdx_forward = (Tn[1:-1, 1:-1, 2:] - Tn[1:-1, 1:-1, 1:-1]) / dx

    dTdy_backward = (Tn[1:-1, 1:-1, 1:-1] - Tn[1:-1, :-2, 1:-1]) / dy
    dTdy_forward = (Tn[1:-1, 2:, 1:-1] - Tn[1:-1, 1:-1, 1:-1]) / dy

    dTdz_backward = (Tn[1:-1, 1:-1, 1:-1] - Tn[:-2, 1:-1, 1:-1]) / dz
    dTdz_forward = (Tn[2:, 1:-1, 1:-1] - Tn[1:-1, 1:-1, 1:-1]) / dz

    dT_dx = np.where(u[1:-1, 1:-1, 1:-1] >= 0.0, dTdx_backward, dTdx_forward)
    dT_dy = np.where(v[1:-1, 1:-1, 1:-1] >= 0.0, dTdy_backward, dTdy_forward)
    dT_dz = np.where(w[1:-1, 1:-1, 1:-1] >= 0.0, dTdz_backward, dTdz_forward)

    lap_T = (
        (Tn[1:-1, 1:-1, 2:] - 2.0 * Tn[1:-1, 1:-1, 1:-1] + Tn[1:-1, 1:-1, :-2]) / dx**2
        +
        (Tn[1:-1, 2:, 1:-1] - 2.0 * Tn[1:-1, 1:-1, 1:-1] + Tn[1:-1, :-2, 1:-1]) / dy**2
        +
        (Tn[2:, 1:-1, 1:-1] - 2.0 * Tn[1:-1, 1:-1, 1:-1] + Tn[:-2, 1:-1, 1:-1]) / dz**2
    )

    T_candidate = Tn[1:-1, 1:-1, 1:-1] + dt * (
        -(u[1:-1, 1:-1, 1:-1] * dT_dx
          + v[1:-1, 1:-1, 1:-1] * dT_dy
          + w[1:-1, 1:-1, 1:-1] * dT_dz)
        +
        alpha * lap_T
        +
        Q_T / (rho_air * cp_air)
    )

    T[1:-1, 1:-1, 1:-1] = np.where(
        fluid_interior,
        T_candidate,
        T[1:-1, 1:-1, 1:-1]
    )

    T = apply_temperature_bcs(T, solid_mask, body_mask)

    return T

## 8. 3D Lagrangian particles

Particles are now released from the upper z face and move downward toward the lower z face.

Particle position equations:

```text
dxp/dt = vpx
dyp/dt = vpy
dzp/dt = vpz
```

Particle velocity equations:

```text
dvpx/dt = (u_air - vpx) / tau_p + gx
dvpy/dt = (v_air - vpy) / tau_p + gy
dvpz/dt = (w_air - vpz) / tau_p + gz
```

where:

```text
xp, yp, zp            = particle position
vpx, vpy, vpz         = particle velocity
u_air, v_air, w_air   = interpolated air velocity
tau_p                 = particle response time
gx, gy, gz            = particle acceleration
```

In this simulation, the particle acceleration follows the new flow direction:

```text
gx = 0
gy = 0
gz = -9.81 m/s²
```

So particles are accelerated from the upper z face toward the lower z face.

Particles can be:

```text
0 = unreleased
1 = active
2 = escaped through outlet strip
3 = deposited on wall
4 = deposited on obstacle
```

In [58]:
def compute_particle_response_time(particle_diameter):
    """
    Compute particle response time using simplified Stokes drag:

        tau_p = rho_p d_p^2 / (18 mu)
    """

    tau_p = (
        particle_density
        *
        particle_diameter**2
        /
        (18.0 * air_dynamic_viscosity)
    )

    return np.maximum(tau_p, 1.0e-12)


def trilinear_interpolate_velocity(xp, yp, zp, u, v, w):
    """
    Interpolate 3D velocity to particle locations.
    """

    xi = xp / dx
    yi = yp / dy
    zi = zp / dz

    i0 = np.floor(xi).astype(int)
    j0 = np.floor(yi).astype(int)
    k0 = np.floor(zi).astype(int)

    i0 = np.clip(i0, 0, nx - 2)
    j0 = np.clip(j0, 0, ny - 2)
    k0 = np.clip(k0, 0, nz - 2)

    i1 = i0 + 1
    j1 = j0 + 1
    k1 = k0 + 1

    sx = xi - i0
    sy = yi - j0
    sz = zi - k0

    def interp(field):
        c000 = field[k0, j0, i0]
        c100 = field[k0, j0, i1]
        c010 = field[k0, j1, i0]
        c110 = field[k0, j1, i1]

        c001 = field[k1, j0, i0]
        c101 = field[k1, j0, i1]
        c011 = field[k1, j1, i0]
        c111 = field[k1, j1, i1]

        return (
            c000 * (1 - sx) * (1 - sy) * (1 - sz)
            + c100 * sx * (1 - sy) * (1 - sz)
            + c010 * (1 - sx) * sy * (1 - sz)
            + c110 * sx * sy * (1 - sz)
            + c001 * (1 - sx) * (1 - sy) * sz
            + c101 * sx * (1 - sy) * sz
            + c011 * (1 - sx) * sy * sz
            + c111 * sx * sy * sz
        )

    up = interp(u)
    vp = interp(v)
    wp = interp(w)

    return up, vp, wp


def inside_fixed_obstacle_particles_3d(xp, yp, zp):
    """
    Check whether particles hit imported CAD obstacle.
    Uses nearest grid point in fixed_obstacle mask.
    """

    i = np.clip(np.round(xp / dx).astype(int), 0, nx - 1)
    j = np.clip(np.round(yp / dy).astype(int), 0, ny - 1)
    k = np.clip(np.round(zp / dz).astype(int), 0, nz - 1)

    return fixed_obstacle[k, j, i]


def inside_moving_body_particles_3d(xp, yp, zp, body_bounds, body_active):
    """
    Check whether particles are inside the moving rectangular body.
    """

    if not body_active or body_bounds is None:
        return np.zeros_like(xp, dtype=bool)

    x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

    inside_body = (
        (xp >= x_min_body)
        &
        (xp <= x_max_body)
        &
        (yp >= y_min_body)
        &
        (yp <= y_max_body)
        &
        (zp >= z_min_body)
        &
        (zp <= z_max_body)
    )

    return inside_body


def release_particles_3d(
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_diameter,
    particle_tau,
    particle_body_offset_x,
    particle_body_offset_y,
    particle_body_offset_z,
    particle_status,
    next_particle_id
):
    """
    Release particles uniformly over the circular inlet disk on the upper z face.
    """

    for _ in range(particles_per_release):
        if next_particle_id >= max_particles:
            break

        r = inlet_radius * np.sqrt(rng.random())
        theta = 2.0 * np.pi * rng.random()

        particle_x[next_particle_id] = inlet_center_x + r * np.cos(theta)
        particle_y[next_particle_id] = inlet_center_y + r * np.sin(theta)
        particle_z[next_particle_id] = particle_release_z

        particle_vx[next_particle_id] = 0.0
        particle_vy[next_particle_id] = 0.0
        particle_vz[next_particle_id] = -U_in

        selected_diameter = rng.choice(
            particle_diameter_options,
            p=particle_diameter_probabilities
        )

        particle_diameter[next_particle_id] = selected_diameter
        particle_tau[next_particle_id] = compute_particle_response_time(selected_diameter)

        particle_body_offset_x[next_particle_id] = np.nan
        particle_body_offset_y[next_particle_id] = np.nan
        particle_body_offset_z[next_particle_id] = np.nan

        particle_status[next_particle_id] = 1
        next_particle_id += 1

    return (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status,
        next_particle_id
    )


def release_stuck_particles_when_body_leaves(
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_status
):
    """
    If particles are stuck on the moving body and the body leaves the room,
    release those particles back into the airflow and let them fall downward.
    """

    stuck = particle_status == 5

    if np.any(stuck):
        particle_x[stuck] = np.clip(
            particle_x[stuck],
            moving_body_release_margin,
            Lx - moving_body_release_margin
        )

        particle_y[stuck] = np.clip(
            particle_y[stuck],
            moving_body_release_margin,
            Ly - moving_body_release_margin
        )

        particle_z[stuck] = np.clip(
            particle_z[stuck],
            moving_body_release_margin,
            Lz - moving_body_release_margin
        )

        particle_vx[stuck] = 0.0
        particle_vy[stuck] = 0.0
        particle_vz[stuck] = particle_fall_release_vz

        particle_status[stuck] = 1

    return (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_status
    )


def step_particles_3d(
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_diameter,
    particle_tau,
    particle_body_offset_x,
    particle_body_offset_y,
    particle_body_offset_z,
    particle_status,
    u,
    v,
    w,
    body_bounds,
    body_velocity_x,
    body_active
):
    """
    3D inertial Lagrangian particle update.

    Particle status:
        0 = unreleased
        1 = active
        2 = escaped
        3 = wall deposited
        4 = CAD obstacle deposited
        5 = stuck on moving body
    """

    dtp = dt / particle_substeps

    for _ in range(particle_substeps):

        # -----------------------------
        # Move stuck particles with moving body
        # -----------------------------
        stuck = particle_status == 5

        if np.any(stuck):
            if body_active and body_bounds is not None:
                x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

                particle_x[stuck] = x_min_body + particle_body_offset_x[stuck]
                particle_y[stuck] = y_min_body + particle_body_offset_y[stuck]
                particle_z[stuck] = z_min_body + particle_body_offset_z[stuck]

                particle_vx[stuck] = body_velocity_x
                particle_vy[stuck] = 0.0
                particle_vz[stuck] = 0.0

            else:
                (
                    particle_x,
                    particle_y,
                    particle_z,
                    particle_vx,
                    particle_vy,
                    particle_vz,
                    particle_status
                ) = release_stuck_particles_when_body_leaves(
                    particle_x,
                    particle_y,
                    particle_z,
                    particle_vx,
                    particle_vy,
                    particle_vz,
                    particle_status
                )

        # -----------------------------
        # Move active particles
        # -----------------------------
        active = particle_status == 1

        if not np.any(active):
            continue

        xp = particle_x[active]
        yp = particle_y[active]
        zp = particle_z[active]

        vpx = particle_vx[active]
        vpy = particle_vy[active]
        vpz = particle_vz[active]

        taup = particle_tau[active]

        up, vp, wp = trilinear_interpolate_velocity(xp, yp, zp, u, v, w)

        exp_factor = np.exp(-dtp / taup)

        terminal_vx = up + taup * g_particle_x
        terminal_vy = vp + taup * g_particle_y
        terminal_vz = wp + taup * g_particle_z

        vpx_new = terminal_vx + (vpx - terminal_vx) * exp_factor
        vpy_new = terminal_vy + (vpy - terminal_vy) * exp_factor
        vpz_new = terminal_vz + (vpz - terminal_vz) * exp_factor

        xp_new = xp + vpx_new * dtp
        yp_new = yp + vpy_new * dtp
        zp_new = zp + vpz_new * dtp

        escaped = (
            (zp_new <= 0.0)
            &
            (xp_new >= outlet_x_min)
            &
            (xp_new <= outlet_x_max)
            &
            (yp_new >= outlet_y_min)
            &
            (yp_new <= outlet_y_max)
        )

        wall_deposit = (
            (xp_new <= 0.0)
            |
            (xp_new >= Lx)
            |
            (yp_new <= 0.0)
            |
            (yp_new >= Ly)
            |
            (zp_new >= Lz)
            |
            (
                (zp_new <= 0.0)
                &
                ~escaped
            )
        )

        cad_obstacle_deposit = inside_fixed_obstacle_particles_3d(
            xp_new,
            yp_new,
            zp_new
        )

        moving_body_contact = inside_moving_body_particles_3d(
            xp_new,
            yp_new,
            zp_new,
            body_bounds,
            body_active
        )

        particle_x[active] = xp_new
        particle_y[active] = yp_new
        particle_z[active] = zp_new

        particle_vx[active] = vpx_new
        particle_vy[active] = vpy_new
        particle_vz[active] = vpz_new

        active_indices = np.where(active)[0]

        particle_status[active_indices[escaped]] = 2
        particle_status[active_indices[wall_deposit]] = 3
        particle_status[active_indices[cad_obstacle_deposit]] = 4

        moving_body_indices = active_indices[moving_body_contact]

        if body_active and body_bounds is not None and len(moving_body_indices) > 0:
            x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

            particle_status[moving_body_indices] = 5

            particle_body_offset_x[moving_body_indices] = (
                particle_x[moving_body_indices] - x_min_body
            )

            particle_body_offset_y[moving_body_indices] = (
                particle_y[moving_body_indices] - y_min_body
            )

            particle_body_offset_z[moving_body_indices] = (
                particle_z[moving_body_indices] - z_min_body
            )

    return (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status
    )

## 9. Run the 3D simulation

This cell runs the full 3D simulation:

```text
1. Solve 3D incompressible airflow
2. Solve 3D temperature transport
3. Release particles at the inlet
4. Move particles through the 3D airflow
5. Store temperature frames and particle frames
```

3D is much heavier than 2D.

For the first test, you can temporarily use:

```python
nt = 100
```

After confirming the code works, change back to:

```python
nt = 1800
```

In [59]:
# -----------------------------
# Helper function: estimate particle no-motion steady-state time
# -----------------------------
def estimate_particle_no_motion_time(
    current_time,
    particle_x,
    particle_y,
    particle_z,
    particle_vx,
    particle_vy,
    particle_vz,
    particle_status,
    next_particle_id,
    body_active
):
    """
    Estimate when particle motion will stop.

    Particle no-motion steady state means:
        all particles have been released
        and no active/stuck particles remain

    This estimate is approximate.

    For active particles:
        estimate time to hit floor/outlet using z / downward speed

    For unreleased particles:
        estimate the future release time, then add approximate fall time

    For particles stuck on moving body:
        estimate at least until the body exits, then add approximate fall time
    """

    remaining_times = []

    # -----------------------------
    # Active particles
    # -----------------------------
    active = particle_status == 1

    if np.any(active):
        z_active = particle_z[active]
        vz_active = particle_vz[active]

        downward_speed = np.maximum(-vz_active, particle_speed_floor)

        time_to_floor = z_active / downward_speed

        remaining_times.extend(time_to_floor.tolist())

    # -----------------------------
    # Stuck particles on moving body
    # -----------------------------
    stuck = particle_status == 5

    if np.any(stuck):
        if body_active:
            time_until_body_exits = max(moving_body_exit_time - current_time, 0.0)
        else:
            time_until_body_exits = 0.0

        z_stuck = particle_z[stuck]
        estimated_fall_speed = max(abs(particle_fall_release_vz), particle_speed_floor)

        time_to_fall_after_release = z_stuck / estimated_fall_speed

        remaining_times.extend(
            (time_until_body_exits + time_to_fall_after_release).tolist()
        )

    # -----------------------------
    # Unreleased particles
    # -----------------------------
    unreleased_count = max_particles - next_particle_id

    if unreleased_count > 0:
        # Future releases happen every release_every steps.
        # Estimate the last future release time.
        steps_needed_to_release_all = unreleased_count * release_every
        time_until_last_release = steps_needed_to_release_all * dt

        # Simple fall-time estimate from inlet height.
        estimated_fall_speed = max(U_in, outlet_velocity, particle_speed_floor)
        estimated_fall_time_from_inlet = particle_release_z / estimated_fall_speed

        remaining_times.append(
            time_until_last_release + estimated_fall_time_from_inlet
        )

    # -----------------------------
    # Estimate final no-motion time
    # -----------------------------
    if len(remaining_times) == 0:
        estimated_time = current_time
    else:
        estimated_time = current_time + particle_estimate_safety_factor * max(remaining_times)

    return estimated_time


# -----------------------------
# Reset fields
# -----------------------------
u_field = np.zeros((nz, ny, nx))
v_field = np.zeros((nz, ny, nx))
w_field = np.zeros((nz, ny, nx))
p = np.zeros((nz, ny, nx))

T_field = T_initial * np.ones((nz, ny, nx))
T_field[fixed_obstacle] = T_aluminum

# Initial moving body
body_mask, body_bounds, body_velocity_x, body_active = get_moving_body_mask(0.0)
solid_mask = fixed_obstacle | body_mask
fluid = ~solid_mask

if body_active:
    T_field[body_mask] = T_body

u_field, v_field, w_field = apply_velocity_bcs(
    u_field,
    v_field,
    w_field,
    solid_mask,
    body_mask,
    body_velocity_x
)

T_field = apply_temperature_bcs(
    T_field,
    solid_mask,
    body_mask
)

# -----------------------------
# Particle arrays
# -----------------------------
particle_x = np.full(max_particles, np.nan)
particle_y = np.full(max_particles, np.nan)
particle_z = np.full(max_particles, np.nan)

particle_vx = np.zeros(max_particles)
particle_vy = np.zeros(max_particles)
particle_vz = np.zeros(max_particles)

particle_diameter = np.full(max_particles, np.nan)
particle_tau = np.full(max_particles, np.nan)

particle_body_offset_x = np.full(max_particles, np.nan)
particle_body_offset_y = np.full(max_particles, np.nan)
particle_body_offset_z = np.full(max_particles, np.nan)

# 0 = unreleased
# 1 = active
# 2 = escaped
# 3 = wall deposited
# 4 = fixed obstacle deposited
# 5 = stuck on moving body
particle_status = np.zeros(max_particles, dtype=int)

next_particle_id = 0

# -----------------------------
# Storage for animation
# -----------------------------
stored_T = []

stored_particle_x = []
stored_particle_y = []
stored_particle_z = []
stored_particle_status = []
stored_particle_diameter = []

stored_body_bounds = []
stored_body_velocity_x = []
stored_body_active = []

# -----------------------------
# Storage for steady-state calculation
# -----------------------------
previous_T_for_steady = T_field.copy()

temperature_steady_counter = 0
particle_steady_counter = 0

temperature_steady_time = None
particle_steady_time = None
overall_steady_time = None

estimated_particle_steady_time = None

temperature_residual_history = []
particle_steady_history = []
particle_estimate_history = []

print("Running 3D CFD + temperature + Lagrangian particle simulation...")

for n in tqdm(range(nt), desc="3D simulation progress", unit="step"):

    current_time = (n + 1) * dt

    # Update moving body
    body_mask, body_bounds, body_velocity_x, body_active = get_moving_body_mask(current_time)
    solid_mask = fixed_obstacle | body_mask
    fluid = ~solid_mask

    # 1. Velocity-pressure update
    u_field, v_field, w_field, p = step_velocity_pressure_3d(
        u_field,
        v_field,
        w_field,
        p,
        solid_mask,
        body_mask,
        body_velocity_x
    )

    # 2. Temperature transport
    T_field = step_temperature_3d(
        T_field,
        u_field,
        v_field,
        w_field,
        solid_mask,
        body_mask
    )

    # 3. Release particles
    if n % release_every == 0:
        (
            particle_x,
            particle_y,
            particle_z,
            particle_vx,
            particle_vy,
            particle_vz,
            particle_diameter,
            particle_tau,
            particle_body_offset_x,
            particle_body_offset_y,
            particle_body_offset_z,
            particle_status,
            next_particle_id
        ) = release_particles_3d(
            particle_x,
            particle_y,
            particle_z,
            particle_vx,
            particle_vy,
            particle_vz,
            particle_diameter,
            particle_tau,
            particle_body_offset_x,
            particle_body_offset_y,
            particle_body_offset_z,
            particle_status,
            next_particle_id
        )

    # 4. Move particles
    (
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status
    ) = step_particles_3d(
        particle_x,
        particle_y,
        particle_z,
        particle_vx,
        particle_vy,
        particle_vz,
        particle_diameter,
        particle_tau,
        particle_body_offset_x,
        particle_body_offset_y,
        particle_body_offset_z,
        particle_status,
        u_field,
        v_field,
        w_field,
        body_bounds,
        body_velocity_x,
        body_active
    )

    # 5. Store frames
    if n % frame_interval == 0:
        stored_T.append(T_field.copy())

        stored_particle_x.append(particle_x.copy())
        stored_particle_y.append(particle_y.copy())
        stored_particle_z.append(particle_z.copy())
        stored_particle_status.append(particle_status.copy())
        stored_particle_diameter.append(particle_diameter.copy())

        stored_body_bounds.append(body_bounds)
        stored_body_velocity_x.append(body_velocity_x)
        stored_body_active.append(body_active)

    # 6. Steady-state and estimate check
    if (n + 1) % steady_check_interval == 0:

        check_time_interval = steady_check_interval * dt

        # -----------------------------
        # Condition 1:
        # Temperature steady state, dT/dt ≈ 0
        # -----------------------------
        T_change = np.abs(T_field - previous_T_for_steady)

        max_temperature_change_rate = np.nanmax(
            T_change[fluid]
        ) / check_time_interval

        mean_temperature_change_rate = np.nanmean(
            T_change[fluid]
        ) / check_time_interval

        temperature_residual_history.append(
            (
                current_time,
                max_temperature_change_rate,
                mean_temperature_change_rate
            )
        )

        temperature_is_steady = (
            max_temperature_change_rate < temperature_steady_tolerance
        )

        if temperature_is_steady:
            temperature_steady_counter += 1
        else:
            temperature_steady_counter = 0

        if (
            temperature_steady_time is None
            and temperature_steady_counter >= temperature_steady_required_checks
        ):
            temperature_steady_time = current_time

            tqdm.write(
                f"Temperature steady state detected at "
                f"t = {temperature_steady_time:.3f} s, "
                f"max dT/dt = {max_temperature_change_rate:.3e} K/s"
            )

        previous_T_for_steady = T_field.copy()

        # -----------------------------
        # Condition 2:
        # Particle no-motion steady state
        # -----------------------------
        released_count = np.sum(particle_status != 0)
        active_count = np.sum(particle_status == 1)
        escaped_count = np.sum(particle_status == 2)
        wall_count = np.sum(particle_status == 3)
        fixed_obstacle_count = np.sum(particle_status == 4)
        stuck_body_count = np.sum(particle_status == 5)
        stopped_count = escaped_count + wall_count + fixed_obstacle_count

        all_particles_released = next_particle_id >= max_particles
        no_particles_moving = (
            active_count == 0
            and
            stuck_body_count == 0
        )

        particle_motion_is_steady = (
            all_particles_released
            and
            no_particles_moving
        )

        # -----------------------------
        # Estimate particle no-motion steady-state time
        # -----------------------------
        estimated_particle_steady_time = estimate_particle_no_motion_time(
            current_time=current_time,
            particle_x=particle_x,
            particle_y=particle_y,
            particle_z=particle_z,
            particle_vx=particle_vx,
            particle_vy=particle_vy,
            particle_vz=particle_vz,
            particle_status=particle_status,
            next_particle_id=next_particle_id,
            body_active=body_active
        )

        particle_estimate_history.append(
            (
                current_time,
                estimated_particle_steady_time
            )
        )

        particle_steady_history.append(
            (
                current_time,
                released_count,
                active_count,
                escaped_count,
                wall_count,
                fixed_obstacle_count,
                stuck_body_count,
                particle_motion_is_steady,
                estimated_particle_steady_time
            )
        )

        if particle_motion_is_steady:
            particle_steady_counter += 1
        else:
            particle_steady_counter = 0

        if (
            particle_steady_time is None
            and particle_steady_counter >= particle_steady_required_checks
        ):
            particle_steady_time = current_time

            tqdm.write(
                f"Particle no-motion steady state detected at "
                f"t = {particle_steady_time:.3f} s"
            )

        # -----------------------------
        # Overall steady state
        # -----------------------------
        if (
            overall_steady_time is None
            and temperature_steady_time is not None
            and particle_steady_time is not None
        ):
            overall_steady_time = current_time

            tqdm.write(
                f"Overall steady state reached at "
                f"t = {overall_steady_time:.3f} s: "
                f"dT/dt ≈ 0 and no particle motion remains."
            )

        if (
            stop_when_both_steady
            and overall_steady_time is not None
        ):
            tqdm.write(
                f"Stopping simulation early at "
                f"t = {current_time:.3f} s."
            )
            break

    # 7. Status output
    if n % 100 == 0:
        speed_now = np.sqrt(u_field**2 + v_field**2 + w_field**2)
        max_speed = np.nanmax(speed_now[fluid])

        actual_CFL_x = max_speed * dt / dx
        actual_CFL_y = max_speed * dt / dy
        actual_CFL_z = max_speed * dt / dz

        max_T = np.nanmax(T_field[fluid]) - 273.15
        min_T = np.nanmin(T_field[fluid]) - 273.15

        released_count = np.sum(particle_status != 0)
        active_count = np.sum(particle_status == 1)
        escaped_count = np.sum(particle_status == 2)
        wall_count = np.sum(particle_status == 3)
        fixed_obstacle_count = np.sum(particle_status == 4)
        stuck_body_count = np.sum(particle_status == 5)

        if len(temperature_residual_history) > 0:
            latest_temperature_rate = temperature_residual_history[-1][1]
        else:
            latest_temperature_rate = np.nan

        if estimated_particle_steady_time is None:
            estimate_text = "not available"
        else:
            estimate_text = f"{estimated_particle_steady_time:.3f} s"

        tqdm.write(
            f"step {n:4d}/{nt}, "
            f"time = {n * dt:.3f} s, "
            f"body active = {body_active}, "
            f"body vx = {body_velocity_x:.2f} m/s, "
            f"max speed = {max_speed:.4f} m/s, "
            f"CFL = ({actual_CFL_x:.3f}, {actual_CFL_y:.3f}, {actual_CFL_z:.3f}), "
            f"T = {min_T:.2f} to {max_T:.2f} °C, "
            f"max dT/dt = {latest_temperature_rate:.3e} K/s, "
            f"released = {released_count}, "
            f"active = {active_count}, "
            f"escaped = {escaped_count}, "
            f"wall dep = {wall_count}, "
            f"fixed obs dep = {fixed_obstacle_count}, "
            f"stuck body = {stuck_body_count}, "
            f"est particle steady = {estimate_text}"
        )

    # 8. Stop if unstable
    active_or_stuck_particles = (particle_status == 1) | (particle_status == 5)

    if (
        not np.isfinite(u_field).all()
        or not np.isfinite(v_field).all()
        or not np.isfinite(w_field).all()
        or not np.isfinite(T_field).all()
        or not np.isfinite(particle_x[active_or_stuck_particles]).all()
        or not np.isfinite(particle_y[active_or_stuck_particles]).all()
        or not np.isfinite(particle_z[active_or_stuck_particles]).all()
    ):
        print(f"Simulation stopped early at step {n} due to NaN or infinity.")
        break

print("Simulation finished.")
print()

print("Stored frames:", len(stored_T))
print("Particles released:", np.sum(particle_status != 0))
print("Particles active:", np.sum(particle_status == 1))
print("Particles escaped:", np.sum(particle_status == 2))
print("Particles deposited on wall:", np.sum(particle_status == 3))
print("Particles deposited on fixed obstacle:", np.sum(particle_status == 4))
print("Particles stuck on moving body:", np.sum(particle_status == 5))

released = particle_status != 0

if np.any(released):
    d_released = particle_diameter[released]

    print()
    print("Released particle diameter counts")
    print(f"0.3 µm: {np.sum(np.isclose(d_released, 0.3e-6))}")
    print(f"0.5 µm: {np.sum(np.isclose(d_released, 0.5e-6))}")
    print(f"1.0 µm: {np.sum(np.isclose(d_released, 1.0e-6))}")

print()
print("Steady-state results")

if temperature_steady_time is not None:
    print(f"Temperature steady-state time: {temperature_steady_time:.3f} s")
else:
    print("Temperature steady state was not reached within this simulation time.")

if particle_steady_time is not None:
    print(f"Particle no-motion steady-state time: {particle_steady_time:.3f} s")
else:
    print("Particle no-motion steady state was not reached within this simulation time.")

    if estimated_particle_steady_time is not None:
        print(f"Estimated particle no-motion steady-state time: {estimated_particle_steady_time:.3f} s")

        suggested_nt = int(np.ceil(estimated_particle_steady_time / dt))
        suggested_nt_with_margin = int(np.ceil(1.20 * estimated_particle_steady_time / dt))

        print(f"Estimated minimum nt needed: {suggested_nt}")
        print(f"Suggested nt with 20% margin: {suggested_nt_with_margin}")

if overall_steady_time is not None:
    print(f"Overall steady-state time: {overall_steady_time:.3f} s")
else:
    print("Overall steady state was not reached within this simulation time.")
    print("Overall condition: dT/dt ≈ 0 and no particle motion remains.")

if len(temperature_residual_history) > 0:
    final_temperature_rate = temperature_residual_history[-1][1]
    print(f"Final max temperature change rate: {final_temperature_rate:.3e} K/s")

if estimated_particle_steady_time is not None:
    print()
    print("Particle steady-state estimate note:")
    print("This is an approximation based on current active particle positions and speeds.")
    print("Use it to tune nt, but confirm by running the simulation longer.")

Running 3D CFD + temperature + Lagrangian particle simulation...


3D simulation progress:   0%|          | 0/1000 [00:00<?, ?step/s]

step    0/1000, time = 0.000 s, body active = False, body vx = 0.00 m/s, max speed = 2.0000 m/s, CFL = (0.600, 0.600, 0.600), T = 25.00 to 35.00 °C, max dT/dt = nan K/s, released = 1, active = 1, escaped = 0, wall dep = 0, fixed obs dep = 0, stuck body = 0, est particle steady = not available
step  100/1000, time = 3.000 s, body active = True, body vx = 1.50 m/s, max speed = 2.5364 m/s, CFL = (0.761, 0.761, 0.761), T = 25.00 to 36.42 °C, max dT/dt = 1.833e+01 K/s, released = 21, active = 19, escaped = 0, wall dep = 0, fixed obs dep = 2, stuck body = 0, est particle steady = 85.461 s
step  200/1000, time = 6.000 s, body active = False, body vx = 0.00 m/s, max speed = 2.0000 m/s, CFL = (0.600, 0.600, 0.600), T = 25.00 to 35.00 °C, max dT/dt = 5.472e+00 K/s, released = 41, active = 34, escaped = 0, wall dep = 0, fixed obs dep = 7, stuck body = 0, est particle steady = 150.191 s
step  300/1000, time = 9.000 s, body active = False, body vx = 0.00 m/s, max speed = 2.0000 m/s, CFL = (0.600, 0

## 10. PyVista helper functions

These functions create the 3D boundary geometry:

```text
room box
top rectangular obstacle
bottom rectangular obstacle
circular inlet disk
circular outlet disk
```

They also help plot particle point clouds.

In [60]:
def make_boundary_geometry():
    """
    Create PyVista geometry for room boundary, inlet, and outlet.
    """

    domain_box = pv.Box(bounds=(0, Lx, 0, Ly, 0, Lz))

    inlet_disk = pv.Disc(
        center=(inlet_center_x, inlet_center_y, Lz),
        inner=0.0,
        outer=inlet_radius,
        normal=(0.0, 0.0, 1.0),
        r_res=1,
        c_res=64
    )

    outlet_strip = pv.Box(
        bounds=(
            outlet_x_min,
            outlet_x_max,
            outlet_y_min,
            outlet_y_max,
            0.0,
            0.02
        )
    )

    return domain_box, inlet_disk, outlet_strip


def make_moving_body_geometry(body_bounds):
    """
    Create PyVista geometry for moving body.
    """

    x_min_body, x_max_body, y_min_body, y_max_body, z_min_body, z_max_body = body_bounds

    body_box = pv.Box(
        bounds=(
            x_min_body,
            x_max_body,
            y_min_body,
            y_max_body,
            z_min_body,
            z_max_body
        )
    )

    return body_box


def add_boundary_to_plotter(plotter):
    """
    Add room boundary, imported CAD boundary, inlet, and outlet.
    """

    domain_box, inlet_disk, outlet_strip = make_boundary_geometry()

    plotter.add_mesh(
        domain_box,
        style="wireframe",
        color="black",
        line_width=1,
        label="4 m selected domain"
    )

    plotter.add_mesh(
        cad_obstacle_visual_mesh,
        color="silver",
        opacity=0.45,
        label="Imported CAD boundary"
    )

    plotter.add_mesh(
        inlet_disk,
        color="green",
        opacity=0.8,
        label="Upper inlet"
    )

    plotter.add_mesh(
        outlet_strip,
        color="blue",
        opacity=0.8,
        label="Floor outlet strip"
    )


def add_or_update_moving_body(plotter, body_bounds, body_active):
    """
    Add, replace, or remove moving warm body.
    """

    if "moving_body" in plotter.actors:
        plotter.remove_actor("moving_body")

    if body_active and body_bounds is not None:
        body_box = make_moving_body_geometry(body_bounds)

        plotter.add_mesh(
            body_box,
            name="moving_body",
            color="orange",
            opacity=0.75,
            label="Moving warm body"
        )


def add_points_if_any(plotter, points, name, color, point_size):
    """
    Add or replace particle point cloud.
    """

    if name in plotter.actors:
        plotter.remove_actor(name)

    if points.shape[0] > 0:
        plotter.add_points(
            points,
            name=name,
            color=color,
            point_size=point_size,
            render_points_as_spheres=True
        )

## 11. Animation 1: particle movement + physical boundary setup

This animation shows only:

```text
room boundary
rectangular obstacles
inlet disk
outlet disk
active particles
deposited particles
```

It does not show the temperature field.

In [61]:
save_folder = Path(r"C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation")
save_folder.mkdir(parents=True, exist_ok=True)

particle_gif_path = save_folder / "3d_particles_CAD.gif"

plotter = pv.Plotter(
    off_screen=True,
    window_size=(1000, 800)
)

add_boundary_to_plotter(plotter)

plotter.add_axes()
plotter.add_legend()
plotter.camera_position = "iso"
plotter.camera.zoom(1.15)

plotter.open_gif(str(particle_gif_path), fps=12)

for frame in tqdm(range(len(stored_particle_x)), desc="Saving 3D particle GIF", unit="frame"):

    px = stored_particle_x[frame]
    py = stored_particle_y[frame]
    pz = stored_particle_z[frame]
    ps = stored_particle_status[frame]

    body_bounds = stored_body_bounds[frame]
    body_active = stored_body_active[frame]

    add_or_update_moving_body(plotter, body_bounds, body_active)

    active = ps == 1
    wall_dep = ps == 3
    fixed_obs_dep = ps == 4
    stuck_body = ps == 5

    active_points = np.column_stack((px[active], py[active], pz[active]))
    wall_points = np.column_stack((px[wall_dep], py[wall_dep], pz[wall_dep]))
    fixed_obstacle_points = np.column_stack((px[fixed_obs_dep], py[fixed_obs_dep], pz[fixed_obs_dep]))
    stuck_body_points = np.column_stack((px[stuck_body], py[stuck_body], pz[stuck_body]))

    # Visual marker sizes are unchanged.
    add_points_if_any(
        plotter,
        active_points,
        "active_particles",
        "yellow",
        8
    )

    add_points_if_any(
        plotter,
        wall_points,
        "wall_deposited_particles",
        "red",
        10
    )

    add_points_if_any(
        plotter,
        fixed_obstacle_points,
        "fixed_obstacle_deposited_particles",
        "red",
        10
    )

    add_points_if_any(
        plotter,
        stuck_body_points,
        "stuck_on_moving_body_particles",
        "purple",
        8
    )

    plotter.add_text(
        f"3D particles + one-pass moving body | frame {frame + 1}/{len(stored_particle_x)}",
        name="frame_text",
        position="upper_left",
        font_size=10
    )

    plotter.write_frame()

plotter.close()

print(f"Saved particle animation to: {particle_gif_path.resolve()}")

Saving 3D particle GIF:   0%|          | 0/67 [00:00<?, ?frame/s]

Saved particle animation to: C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation\3d_particles_CAD.gif


## 12. Animation 2: temperature field + physical boundary setup

This animation shows:

```text
room boundary
floor-attached rectangular obstacles
upper inlet disk
lower outlet strip
temperature field slice in x direction and in y direction
```

It does not show particles.

Because the main flow direction is now from upper z face to lower z face, this animation uses a vertical center slice:

```text
y = Ly / 2
```

This makes it easier to see temperature moving from top to bottom.

In [62]:
temperature_gif_path = save_folder / "3d_temperature_CAD_along_x.gif"

# PyVista ImageData grid
grid = pv.ImageData(
    dimensions=(nx, ny, nz),
    spacing=(dx, dy, dz),
    origin=(0.0, 0.0, 0.0)
)

# Temperature color limits
fixed_fluid_mask = ~fixed_obstacle

all_min = min(float(np.nanmin(T[fixed_fluid_mask] - 273.15)) for T in stored_T)
all_max = max(float(np.nanmax(T[fixed_fluid_mask] - 273.15)) for T in stored_T)

plotter = pv.Plotter(
    off_screen=True,
    window_size=(1000, 800)
)

add_boundary_to_plotter(plotter)

plotter.add_axes()
plotter.add_legend()
plotter.camera_position = "iso"
plotter.camera.zoom(1.15)

plotter.open_gif(str(temperature_gif_path), fps=12)

for frame in tqdm(range(len(stored_T)), desc="Saving 3D temperature GIF", unit="frame"):

    T_c = stored_T[frame].copy() - 273.15
    T_c[fixed_obstacle] = np.nan

    body_bounds = stored_body_bounds[frame]
    body_active = stored_body_active[frame]

    add_or_update_moving_body(plotter, body_bounds, body_active)

    # Convert from [z, y, x] to [x, y, z] for PyVista/VTK
    T_xyz = np.transpose(T_c, (2, 1, 0))

    grid.point_data["Temperature_C"] = T_xyz.ravel(order="F")

    # Vertical middle slice at y = Ly / 2
    temp_slice = grid.slice(
        normal="y",
        origin=(0.0, Ly / 2.0, 0.0)
    )

    plotter.add_mesh(
        temp_slice,
        name="temperature_slice",
        scalars="Temperature_C",
        cmap="coolwarm",
        clim=(all_min, all_max),
        opacity=0.9,
        show_scalar_bar=True,
        scalar_bar_args={"title": "Temperature [C]"}
    )

    plotter.add_text(
        f"3D temperature + one-pass moving body | frame {frame + 1}/{len(stored_T)}",
        name="frame_text",
        position="upper_left",
        font_size=10
    )

    plotter.write_frame()

plotter.close()

print(f"Saved temperature animation to: {temperature_gif_path.resolve()}")

Saving 3D temperature GIF:   0%|          | 0/67 [00:00<?, ?frame/s]

Saved temperature animation to: C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation\3d_temperature_CAD_along_x.gif


In [63]:
# ============================================================
# Animation 3: temperature field on fixed y-z plane
# ============================================================

temperature_yz_gif_path = save_folder / "3d_temperature_CAD_along_y.gif"

# PyVista ImageData grid
grid = pv.ImageData(
    dimensions=(nx, ny, nz),
    spacing=(dx, dy, dz),
    origin=(0.0, 0.0, 0.0)
)

# Temperature color limits
fixed_fluid_mask = ~fixed_obstacle

all_min = min(float(np.nanmin(T[fixed_fluid_mask] - 273.15)) for T in stored_T)
all_max = max(float(np.nanmax(T[fixed_fluid_mask] - 273.15)) for T in stored_T)

plotter = pv.Plotter(
    off_screen=True,
    window_size=(1000, 800)
)

add_boundary_to_plotter(plotter)

plotter.add_axes()
plotter.add_legend()
plotter.camera_position = "iso"
plotter.camera.zoom(1.15)

plotter.open_gif(str(temperature_yz_gif_path), fps=12)

# Fixed y-z plane location
# A y-z plane means x = constant
x_slice_position = Lx / 2.0

for frame in tqdm(range(len(stored_T)), desc="Saving y-z temperature GIF", unit="frame"):

    T_c = stored_T[frame].copy() - 273.15
    T_c[fixed_obstacle] = np.nan

    # Moving body update
    body_bounds = stored_body_bounds[frame]

    if "stored_body_active" in globals():
        body_active = stored_body_active[frame]
    else:
        body_active = body_bounds is not None

    add_or_update_moving_body(
        plotter,
        body_bounds,
        body_active
    )

    # Convert from solver shape [z, y, x]
    # to PyVista shape [x, y, z]
    T_xyz = np.transpose(T_c, (2, 1, 0))

    grid.point_data["Temperature_C"] = T_xyz.ravel(order="F")

    # Fixed y-z slice:
    # normal = x means the slice plane is y-z
    temp_slice = grid.slice(
        normal="x",
        origin=(x_slice_position, 0.0, 0.0)
    )

    if "temperature_yz_slice" in plotter.actors:
        plotter.remove_actor("temperature_yz_slice")

    plotter.add_mesh(
        temp_slice,
        name="temperature_yz_slice",
        scalars="Temperature_C",
        cmap="coolwarm",
        clim=(all_min, all_max),
        opacity=0.9,
        show_scalar_bar=True,
        scalar_bar_args={"title": "Temperature [C]"}
    )

    plotter.add_text(
        (
            f"Temperature field y-z plane\n"
            f"frame {frame + 1}/{len(stored_T)} | "
            f"x = {x_slice_position:.2f} m"
        ),
        name="frame_text",
        position="upper_left",
        font_size=10
    )

    plotter.write_frame()

plotter.close()

print(f"Saved y-z temperature animation to: {temperature_yz_gif_path}")

Saving y-z temperature GIF:   0%|          | 0/67 [00:00<?, ?frame/s]

Saved y-z temperature animation to: C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation\3d_temperature_CAD_along_y.gif
